In [1]:
!pip install -q sentence-transformers faiss-cpu groq gradio pandas numpy

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.8/23.8 MB 48.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 142.3/142.3 kB 6.5 MB/s eta 0:00:00


In [2]:
import os
import json
import numpy as np
import pandas as pd
from sentence_transformers import SentenceTransformer
import faiss
import gradio as gr
from groq import Groq

# ✅ Paste your Groq API key here
GROQ_API_KEY = "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"

client = Groq(api_key=GROQ_API_KEY)
print("✅ Libraries loaded and Groq client initialized.")

✅ Libraries loaded and Groq client initialized.


In [3]:
ipc_data = [
    {"section": "IPC 299", "title": "Culpable Homicide", "description": "Whoever causes death by doing an act with the intention of causing death, or with the intention of causing such bodily injury as is likely to cause death, or with the knowledge that he is likely by such act to cause death.", "category": "Offences Against Body", "punishment": "Imprisonment for life or up to 10 years and fine"},
    {"section": "IPC 300", "title": "Murder", "description": "Culpable homicide is murder if the act causing death is done with the intention of causing death, or with the intention of causing bodily injury known to be likely to cause death, or causing bodily injury sufficient to cause death, or doing an imminently dangerous act.", "category": "Offences Against Body", "punishment": "Death or imprisonment for life and fine"},
    {"section": "IPC 302", "title": "Punishment for Murder", "description": "Whoever commits murder shall be punished with death or imprisonment for life and shall also be liable to fine.", "category": "Offences Against Body", "punishment": "Death or life imprisonment and fine"},
    {"section": "IPC 304", "title": "Punishment for Culpable Homicide not amounting to Murder", "description": "Whoever commits culpable homicide not amounting to murder shall be punished with imprisonment for life or imprisonment up to 10 years and fine.", "category": "Offences Against Body", "punishment": "Life imprisonment or up to 10 years and fine"},
    {"section": "IPC 304A", "title": "Causing Death by Negligence", "description": "Whoever causes the death of any person by doing any rash or negligent act not amounting to culpable homicide shall be punished with imprisonment up to 2 years or fine or both.", "category": "Offences Against Body", "punishment": "Up to 2 years imprisonment or fine or both"},
    {"section": "IPC 304B", "title": "Dowry Death", "description": "Where the death of a woman is caused by burns or bodily injury within 7 years of marriage under circumstances showing cruelty or harassment by husband or in-laws in connection with dowry demands.", "category": "Offences Against Body", "punishment": "Minimum 7 years, may extend to life imprisonment"},
    {"section": "IPC 305", "title": "Abetment of Suicide of Child or Insane Person", "description": "If any person under 18, insane, delirious, or in a state of intoxication commits suicide, whoever abets the commission of such suicide shall be punished.", "category": "Offences Against Body", "punishment": "Death or life imprisonment or up to 10 years and fine"},
    {"section": "IPC 306", "title": "Abetment of Suicide", "description": "If any person commits suicide, whoever abets the commission of such suicide shall be punished with imprisonment up to 10 years and fine.", "category": "Offences Against Body", "punishment": "Up to 10 years imprisonment and fine"},
    {"section": "IPC 307", "title": "Attempt to Murder", "description": "Whoever does any act with such intention or knowledge and under such circumstances that if he by that act caused death he would be guilty of murder shall be punished.", "category": "Offences Against Body", "punishment": "Up to 10 years and fine; life if hurt caused"},
    {"section": "IPC 308", "title": "Attempt to Commit Culpable Homicide", "description": "Whoever does any act with such intention or knowledge that if death is caused it would be culpable homicide not amounting to murder shall be punished.", "category": "Offences Against Body", "punishment": "Up to 3 years and fine; up to 7 years if hurt caused"},
    {"section": "IPC 309", "title": "Attempt to Commit Suicide", "description": "Whoever attempts to commit suicide and does any act towards the commission of such offence shall be punished with simple imprisonment up to 1 year or fine or both.", "category": "Offences Against Body", "punishment": "Up to 1 year imprisonment or fine or both"},
    {"section": "IPC 310", "title": "Thug", "description": "Whoever has been habitually associated with any other person or persons for the purpose of committing robbery or child-stealing by means of or accompanied with murder is a thug.", "category": "Offences Against Body", "punishment": "Imprisonment for life and fine"},
    {"section": "IPC 319", "title": "Hurt", "description": "Whoever causes bodily pain, disease or infirmity to any person is said to cause hurt.", "category": "Offences Against Body", "punishment": "Up to 1 year or fine up to 1000 or both"},
    {"section": "IPC 320", "title": "Grievous Hurt", "description": "Grievous hurt includes emasculation, permanent loss of sight or hearing, loss of limb or joint, permanent disfiguration, fracture or dislocation of bone, or hurt endangering life.", "category": "Offences Against Body", "punishment": "Up to 7 years and fine"},
    {"section": "IPC 323", "title": "Punishment for Voluntarily Causing Hurt", "description": "Whoever voluntarily causes hurt shall be punished with imprisonment up to 1 year or fine up to 1000 rupees or both.", "category": "Offences Against Body", "punishment": "Up to 1 year or fine up to Rs.1000 or both"},
    {"section": "IPC 324", "title": "Voluntarily Causing Hurt by Dangerous Weapons", "description": "Whoever voluntarily causes hurt by means of any instrument for shooting, stabbing or cutting, or by means of fire or heated substance, or by poison or corrosive substance.", "category": "Offences Against Body", "punishment": "Up to 3 years or fine or both"},
    {"section": "IPC 325", "title": "Punishment for Voluntarily Causing Grievous Hurt", "description": "Whoever voluntarily causes grievous hurt shall be punished with imprisonment up to 7 years and fine.", "category": "Offences Against Body", "punishment": "Up to 7 years and fine"},
    {"section": "IPC 326", "title": "Voluntarily Causing Grievous Hurt by Dangerous Weapons", "description": "Whoever voluntarily causes grievous hurt by means of any instrument for shooting, stabbing or cutting, or by means of fire, heated substance, poison, or corrosive substance.", "category": "Offences Against Body", "punishment": "Life imprisonment or up to 10 years and fine"},
    {"section": "IPC 354", "title": "Assault or Criminal Force to Woman with Intent to Outrage Modesty", "description": "Whoever assaults or uses criminal force to any woman intending to outrage or knowing it likely to outrage the modesty of that woman shall be punished.", "category": "Offences Against Women", "punishment": "Up to 2 years or fine or both"},
    {"section": "IPC 354A", "title": "Sexual Harassment", "description": "Physical contact and advances involving unwelcome and explicit sexual overtures, demand or request for sexual favors, showing pornography, unwelcome physical, verbal or non-verbal conduct of sexual nature.", "category": "Offences Against Women", "punishment": "Up to 3 years or fine or both"},
    {"section": "IPC 354B", "title": "Assault with Intent to Disrobe a Woman", "description": "Whoever assaults or uses criminal force to any woman or abets such act with the intention of disrobing or compelling her to be naked shall be punished.", "category": "Offences Against Women", "punishment": "3 to 7 years and fine"},
    {"section": "IPC 354C", "title": "Voyeurism", "description": "Any man who watches or captures the image of a woman engaging in a private act in circumstances where she would not expect observation shall be punished.", "category": "Offences Against Women", "punishment": "1 to 3 years first conviction; 3 to 7 years subsequent"},
    {"section": "IPC 354D", "title": "Stalking", "description": "Any man who follows a woman and contacts or attempts to contact to foster personal interaction repeatedly despite disinterest, or monitors use of internet/email by a woman.", "category": "Offences Against Women", "punishment": "Up to 3 years first conviction; up to 5 years subsequent"},
    {"section": "IPC 363", "title": "Punishment for Kidnapping", "description": "Whoever kidnaps any person from India or from lawful guardianship shall be punished with imprisonment up to 7 years and fine.", "category": "Kidnapping and Abduction", "punishment": "Up to 7 years and fine"},
    {"section": "IPC 364", "title": "Kidnapping for Murder", "description": "Whoever kidnaps any person in order that such person may be murdered or so disposed of as to be put in danger of being murdered shall be punished.", "category": "Kidnapping and Abduction", "punishment": "Life imprisonment or up to 10 years and fine"},
    {"section": "IPC 365", "title": "Kidnapping with Intent to Secretly and Wrongfully Confine", "description": "Whoever kidnaps any person with intent to cause that person to be secretly and wrongfully confined shall be punished with imprisonment up to 7 years and fine.", "category": "Kidnapping and Abduction", "punishment": "Up to 7 years and fine"},
    {"section": "IPC 366", "title": "Kidnapping or Abducting a Woman to Compel Marriage", "description": "Whoever kidnaps or abducts any woman with intent that she may be compelled to marry any person against her will, or forced into illicit intercourse shall be punished.", "category": "Offences Against Women", "punishment": "Up to 10 years and fine"},
    {"section": "IPC 376", "title": "Punishment for Rape", "description": "Sexual intercourse by a man with a woman against her will, without her consent, with her consent obtained by threat of death or hurt, or when she is unable to understand the nature of the act.", "category": "Offences Against Women", "punishment": "Minimum 7 years up to life imprisonment and fine"},
    {"section": "IPC 376A", "title": "Punishment for Rape Causing Death or Persistent Vegetative State", "description": "Whoever commits rape and inflicts injury which causes death of the woman or results in a persistent vegetative state of the woman shall be punished.", "category": "Offences Against Women", "punishment": "Minimum 20 years up to life imprisonment or death"},
    {"section": "IPC 376D", "title": "Gang Rape", "description": "Where a woman is raped by one or more persons constituting a group or acting in furtherance of a common intention each person shall be deemed to have committed rape.", "category": "Offences Against Women", "punishment": "Minimum 20 years up to life imprisonment and fine"},
    {"section": "IPC 378", "title": "Theft", "description": "Whoever intending to take dishonestly any moveable property out of the possession of any person without that person's consent moves that property is said to commit theft.", "category": "Offences Against Property", "punishment": "Up to 3 years or fine or both"},
    {"section": "IPC 379", "title": "Punishment for Theft", "description": "Whoever commits theft shall be punished with imprisonment up to 3 years or fine or both.", "category": "Offences Against Property", "punishment": "Up to 3 years or fine or both"},
    {"section": "IPC 380", "title": "Theft in Dwelling House", "description": "Whoever commits theft in any building tent or vessel used as a human dwelling or used for custody of property shall be punished with imprisonment up to 7 years and fine.", "category": "Offences Against Property", "punishment": "Up to 7 years and fine"},
    {"section": "IPC 381", "title": "Theft by Clerk or Servant", "description": "Whoever commits theft having at the time a clerk or servant in the employ of the person from whose possession property is stolen shall be punished.", "category": "Offences Against Property", "punishment": "Up to 7 years and fine"},
    {"section": "IPC 382", "title": "Theft after Preparation for Causing Death or Hurt", "description": "Whoever commits theft having made preparation for causing death or hurt to any person in order to commit such theft shall be punished.", "category": "Offences Against Property", "punishment": "Up to 10 years and fine"},
    {"section": "IPC 383", "title": "Extortion", "description": "Whoever intentionally puts any person in fear of any injury and dishonestly induces the person so put in fear to deliver any property or to commit an act which is illegal commits extortion.", "category": "Offences Against Property", "punishment": "Up to 3 years or fine or both"},
    {"section": "IPC 384", "title": "Punishment for Extortion", "description": "Whoever commits extortion shall be punished with imprisonment up to 3 years or fine or both.", "category": "Offences Against Property", "punishment": "Up to 3 years or fine or both"},
    {"section": "IPC 390", "title": "Robbery", "description": "In all robbery there is either theft or extortion. Theft becomes robbery when the offender uses force or threatens force to commit theft or when carrying away stolen property.", "category": "Offences Against Property", "punishment": "Up to 10 years and fine"},
    {"section": "IPC 391", "title": "Dacoity", "description": "When 5 or more persons conjointly commit or attempt to commit a robbery, or where the whole number of persons conjointly committing or attempting to commit a robbery and abetting are 5 or more, every person so committing is said to commit dacoity.", "category": "Offences Against Property", "punishment": "Life imprisonment or up to 10 years and fine"},
    {"section": "IPC 392", "title": "Punishment for Robbery", "description": "Whoever commits robbery shall be punished with rigorous imprisonment up to 10 years and fine; if on a highway between sunset and sunrise, up to 14 years.", "category": "Offences Against Property", "punishment": "Up to 10 years and fine"},
    {"section": "IPC 395", "title": "Punishment for Dacoity", "description": "Whoever commits dacoity shall be punished with imprisonment for life or rigorous imprisonment up to 10 years and fine.", "category": "Offences Against Property", "punishment": "Life imprisonment or up to 10 years and fine"},
    {"section": "IPC 396", "title": "Dacoity with Murder", "description": "If any one of 5 or more persons who are conjointly committing dacoity commits murder in so committing dacoity, every one of those persons shall be punished with death or life imprisonment.", "category": "Offences Against Property", "punishment": "Death or life imprisonment and fine"},
    {"section": "IPC 406", "title": "Punishment for Criminal Breach of Trust", "description": "Whoever commits criminal breach of trust shall be punished with imprisonment up to 3 years or fine or both.", "category": "Offences Against Property", "punishment": "Up to 3 years or fine or both"},
    {"section": "IPC 415", "title": "Cheating", "description": "Whoever by deceiving any person fraudulently or dishonestly induces the person so deceived to deliver any property or commits any act which the person would not do is said to cheat.", "category": "Offences Against Property", "punishment": "Up to 1 year or fine or both"},
    {"section": "IPC 417", "title": "Punishment for Cheating", "description": "Whoever cheats shall be punished with imprisonment up to 1 year or fine or both.", "category": "Offences Against Property", "punishment": "Up to 1 year or fine or both"},
    {"section": "IPC 418", "title": "Cheating with Knowledge that Wrongful Loss may Ensue", "description": "Whoever cheats with the knowledge that he is likely thereby to cause wrongful loss to a person whose interest in the transaction to which the cheating relates he was bound to protect shall be punished.", "category": "Offences Against Property", "punishment": "Up to 3 years or fine or both"},
    {"section": "IPC 420", "title": "Cheating and Dishonestly Inducing Delivery of Property", "description": "Whoever cheats and thereby dishonestly induces the person deceived to deliver any property or alter or destroy signed or sealed documents shall be punished.", "category": "Offences Against Property", "punishment": "Up to 7 years and fine"},
    {"section": "IPC 425", "title": "Mischief", "description": "Whoever with intent to cause wrongful loss or damage to the public or any person causes the destruction of any property or any such change in the property that destroys or diminishes its value commits mischief.", "category": "Offences Against Property", "punishment": "Up to 3 months or fine or both"},
    {"section": "IPC 426", "title": "Punishment for Mischief", "description": "Whoever commits mischief shall be punished with imprisonment up to 3 months or fine or both.", "category": "Offences Against Property", "punishment": "Up to 3 months or fine or both"},
    {"section": "IPC 435", "title": "Mischief by Fire or Explosive to Damage Property", "description": "Whoever commits mischief by fire or any explosive substance intending to cause damage to any property shall be punished with imprisonment up to 7 years and fine.", "category": "Offences Against Property", "punishment": "Up to 7 years and fine"},
    {"section": "IPC 436", "title": "Mischief by Fire or Explosive to Destroy House", "description": "Whoever commits mischief by fire or any explosive substance intending to destroy any building used as a dwelling or as a place of worship shall be punished with life imprisonment.", "category": "Offences Against Property", "punishment": "Life imprisonment or up to 10 years and fine"},
    {"section": "IPC 441", "title": "Criminal Trespass", "description": "Whoever enters into or upon property in possession of another with intent to commit an offence or intimidate insult or annoy any person in possession of such property commits criminal trespass.", "category": "Offences Against Property", "punishment": "Up to 3 months or fine up to Rs.500 or both"},
    {"section": "IPC 447", "title": "Punishment for Criminal Trespass", "description": "Whoever commits criminal trespass shall be punished with imprisonment up to 3 months or fine up to Rs.500 or both.", "category": "Offences Against Property", "punishment": "Up to 3 months or fine up to Rs.500 or both"},
    {"section": "IPC 448", "title": "Punishment for House-trespass", "description": "Whoever commits house-trespass shall be punished with imprisonment up to 1 year or fine up to Rs.1000 or both.", "category": "Offences Against Property", "punishment": "Up to 1 year or fine up to Rs.1000 or both"},
    {"section": "IPC 454", "title": "Lurking House-trespass or House-breaking to Commit an Offence", "description": "Whoever commits lurking house-trespass or house-breaking in order to commit any offence punishable with imprisonment shall be punished.", "category": "Offences Against Property", "punishment": "Up to 3 years and fine"},
    {"section": "IPC 457", "title": "Lurking House-trespass or House-breaking at Night", "description": "Whoever commits lurking house-trespass by night or house-breaking by night in order to commit any offence punishable with imprisonment shall be punished.", "category": "Offences Against Property", "punishment": "Up to 5 years and fine"},
    {"section": "IPC 463", "title": "Forgery", "description": "Whoever makes any false document or false electronic record or part thereof with intent to cause damage or injury or to support any claim or title or to commit fraud commits forgery.", "category": "Offences Relating to Documents", "punishment": "Up to 2 years or fine or both"},
    {"section": "IPC 465", "title": "Punishment for Forgery", "description": "Whoever commits forgery shall be punished with imprisonment up to 2 years or fine or both.", "category": "Offences Relating to Documents", "punishment": "Up to 2 years or fine or both"},
    {"section": "IPC 468", "title": "Forgery for Purpose of Cheating", "description": "Whoever commits forgery intending that the document or electronic record forged shall be used for the purpose of cheating shall be punished.", "category": "Offences Relating to Documents", "punishment": "Up to 7 years and fine"},
    {"section": "IPC 471", "title": "Using as Genuine a Forged Document", "description": "Whoever fraudulently or dishonestly uses as genuine any document or electronic record which he knows or has reason to believe to be a forged document shall be punished.", "category": "Offences Relating to Documents", "punishment": "Same as for forgery of that document"},
    {"section": "IPC 489A", "title": "Counterfeiting Currency Notes or Bank Notes", "description": "Whoever counterfeits or knowingly performs any part of the process of counterfeiting any currency note or bank note shall be punished with imprisonment for life or imprisonment up to 10 years and fine.", "category": "Offences Relating to Currency", "punishment": "Life imprisonment or up to 10 years and fine"},
    {"section": "IPC 489B", "title": "Using as Genuine Forged or Counterfeit Currency", "description": "Whoever sells or uses as genuine or attempts to use as genuine any forged or counterfeit currency note or bank note knowing to be forged shall be punished.", "category": "Offences Relating to Currency", "punishment": "Life imprisonment or up to 10 years and fine"},
    {"section": "IPC 498A", "title": "Cruelty by Husband or Relatives of Husband", "description": "Whoever being the husband or the relative of the husband of a woman subjects her to cruelty involving wilful conduct likely to drive her to suicide or cause grave injury, or harassment for dowry.", "category": "Offences Against Women", "punishment": "Up to 3 years and fine"},
    {"section": "IPC 499", "title": "Defamation", "description": "Whoever by words makes or publishes any imputation concerning any person intending to harm the reputation of such person is said to defame that person subject to exceptions.", "category": "Offences Against Reputation", "punishment": "Up to 2 years or fine or both"},
    {"section": "IPC 500", "title": "Punishment for Defamation", "description": "Whoever defames another shall be punished with simple imprisonment up to 2 years or fine or both.", "category": "Offences Against Reputation", "punishment": "Up to 2 years or fine or both"},
    {"section": "IPC 504", "title": "Intentional Insult with Intent to Provoke Breach of Peace", "description": "Whoever intentionally insults and thereby gives provocation to any person intending that such provocation will cause him to break public peace or commit any other offence shall be punished.", "category": "Offences Against Public Order", "punishment": "Up to 2 years or fine or both"},
    {"section": "IPC 506", "title": "Punishment for Criminal Intimidation", "description": "Whoever commits criminal intimidation shall be punished with imprisonment up to 2 years or fine or both; if threat be to cause death or grievous hurt up to 7 years.", "category": "Offences Against Public Order", "punishment": "Up to 2 years; up to 7 years if threat of death or grievous hurt"},
    {"section": "IPC 509", "title": "Word Gesture Intended to Insult Modesty of a Woman", "description": "Whoever intending to insult the modesty of any woman utters any word or makes any sound or gesture or exhibits any object intending that such word shall be heard or seen by such woman shall be punished.", "category": "Offences Against Women", "punishment": "Up to 3 years or fine or both"},
    {"section": "IPC 107", "title": "Abetment of a Thing", "description": "A person abets the doing of a thing who instigates any person to do that thing, engages in any conspiracy for doing that thing, or intentionally aids by any act or illegal omission the doing of that thing.", "category": "Abetment", "punishment": "Same punishment as for the offence abetted"},
    {"section": "IPC 109", "title": "Punishment of Abetment if Act Abetted is Committed", "description": "Whoever abets any offence shall if the act abetted is committed in consequence of the abetment and no express provision is made for its punishment be punished with the punishment provided for the offence.", "category": "Abetment", "punishment": "Same as the offence abetted"},
    {"section": "IPC 120A", "title": "Criminal Conspiracy", "description": "When two or more persons agree to do or cause to be done an illegal act or an act which is not illegal by illegal means such an agreement is designated a criminal conspiracy.", "category": "Criminal Conspiracy", "punishment": "Same as offence conspired; up to 6 months if minor offence"},
    {"section": "IPC 120B", "title": "Punishment of Criminal Conspiracy", "description": "Whoever is a party to a criminal conspiracy to commit an offence punishable with death or rigorous imprisonment for 2 years or more shall be punished in the same manner as if he had abetted such offence.", "category": "Criminal Conspiracy", "punishment": "Same as abetment of the offence"},
    {"section": "IPC 141", "title": "Unlawful Assembly", "description": "An assembly of 5 or more persons is designated an unlawful assembly if the common object is to overawe the government, resist legal process, commit mischief or criminal trespass, or dispute possession of property by force.", "category": "Offences Against Public Order", "punishment": "Up to 6 months or fine or both"},
    {"section": "IPC 143", "title": "Punishment for Member of Unlawful Assembly", "description": "Whoever is a member of an unlawful assembly shall be punished with imprisonment up to 6 months or fine or both.", "category": "Offences Against Public Order", "punishment": "Up to 6 months or fine or both"},
    {"section": "IPC 144", "title": "Joining Unlawful Assembly Armed with Deadly Weapon", "description": "Whoever being armed with any deadly weapon or with anything which used as a weapon is likely to cause death joins or continues in any unlawful assembly shall be punished.", "category": "Offences Against Public Order", "punishment": "Up to 2 years or fine or both"},
    {"section": "IPC 147", "title": "Punishment for Rioting", "description": "Whoever is guilty of rioting shall be punished with imprisonment up to 2 years or fine or both.", "category": "Offences Against Public Order", "punishment": "Up to 2 years or fine or both"},
    {"section": "IPC 148", "title": "Rioting Armed with Deadly Weapon", "description": "Whoever is guilty of rioting being armed with a deadly weapon or with anything which used as a weapon is likely to cause death shall be punished with imprisonment up to 3 years or fine or both.", "category": "Offences Against Public Order", "punishment": "Up to 3 years or fine or both"},
    {"section": "IPC 153A", "title": "Promoting Enmity Between Groups", "description": "Whoever by words or signs promotes disharmony or feelings of enmity hatred or ill-will between different religious racial language or regional groups caste or communities or acts prejudicial to maintenance of harmony shall be punished.", "category": "Offences Against Public Tranquility", "punishment": "Up to 3 years or fine or both; up to 5 years if at place of worship"},
    {"section": "IPC 171B", "title": "Bribery in Elections", "description": "Whoever gives a gratification to any person with the object of inducing him or any other person to exercise any electoral right or of rewarding any person for having exercised any such right commits the offence of bribery.", "category": "Offences Relating to Elections", "punishment": "Up to 1 year or fine or both"},
    {"section": "IPC 186", "title": "Obstructing Public Servant in Discharge of Public Function", "description": "Whoever voluntarily obstructs any public servant in the discharge of his public functions shall be punished with imprisonment up to 3 months or fine up to Rs.500 or both.", "category": "Offences Against Public Servants", "punishment": "Up to 3 months or fine up to Rs.500 or both"},
    {"section": "IPC 188", "title": "Disobedience to Order of Public Servant", "description": "Whoever knowingly disobeys any direction lawfully given by a public servant knowing disobedience causes or tends to cause obstruction annoyance or injury or risk of obstruction annoyance or injury to any person shall be punished.", "category": "Offences Against Public Servants", "punishment": "Up to 1 month or fine up to Rs.200 or both"},
    {"section": "IPC 193", "title": "Punishment for False Evidence", "description": "Whoever intentionally gives false evidence in any stage of a judicial proceeding or fabricates false evidence for the purpose of being used in any stage of a judicial proceeding shall be punished.", "category": "Offences Against Public Justice", "punishment": "Up to 7 years and fine"},
    {"section": "IPC 201", "title": "Causing Disappearance of Evidence or Giving False Information", "description": "Whoever knowing or having reason to believe that an offence has been committed causes any evidence of the commission of that offence to disappear or gives false information to screen the offender shall be punished.", "category": "Offences Against Public Justice", "punishment": "Up to 7 years; if capital offence up to life imprisonment"},
    {"section": "IPC 211", "title": "False Charge of Offence Made with Intent to Injure", "description": "Whoever with intent to cause injury institutes or causes to be instituted any criminal proceeding or falsely charges any person with having committed an offence knowing there is no just or lawful ground for such proceeding shall be punished.", "category": "Offences Against Public Justice", "punishment": "Up to 2 years or fine or both; up to 7 years if capital offence"},
    {"section": "IPC 268", "title": "Public Nuisance", "description": "A person is guilty of a public nuisance who does any act or is guilty of an illegal omission which causes common injury danger or annoyance to the public or people in general who dwell or occupy property in the vicinity.", "category": "Offences Affecting Public Health", "punishment": "Fine up to Rs.200"},
    {"section": "IPC 270", "title": "Making Environment Noxious to Health", "description": "Whoever voluntarily corrupts or fouls the water of any public spring or reservoir so as to render it less fit for the purposes for which it is ordinarily used shall be punished.", "category": "Offences Affecting Public Health", "punishment": "Up to 3 months or fine or both"},
    {"section": "IPC 272", "title": "Adulteration of Food or Drink", "description": "Whoever adulterates any article of food or drink so as to make such article noxious as food or drink intending to sell such article or knowing that it is likely to be sold shall be punished.", "category": "Offences Affecting Public Health", "punishment": "Up to 6 months or fine up to Rs.1000 or both"},
    {"section": "IPC 279", "title": "Rash Driving or Riding on Public Way", "description": "Whoever drives any vehicle or rides on any public way in a manner so rash or negligent as to endanger human life or likely to cause hurt or injury to any other person shall be punished.", "category": "Offences Affecting Public Safety", "punishment": "Up to 6 months or fine up to Rs.1000 or both"},
    {"section": "IPC 295", "title": "Injuring or Defiling Place of Worship", "description": "Whoever destroys damages or defiles any place of worship or any object held sacred by any class of persons with the intention of thereby insulting the religion of any class of persons shall be punished.", "category": "Offences Relating to Religion", "punishment": "Up to 2 years or fine or both"},
    {"section": "IPC 295A", "title": "Deliberate Acts to Outrage Religious Feelings", "description": "Whoever with deliberate and malicious intention of outraging the religious feelings of any class by insulting its religion or religious beliefs shall be punished.", "category": "Offences Relating to Religion", "punishment": "Up to 3 years or fine or both"},
    {"section": "IPC 34", "title": "Acts Done by Several Persons in Furtherance of Common Intention", "description": "When a criminal act is done by several persons in furtherance of the common intention of all each of such persons is liable for that act in the same manner as if it were done by him alone.", "category": "General Exceptions", "punishment": "Same as if done individually"},
    {"section": "IPC 84", "title": "Act of a Person of Unsound Mind", "description": "Nothing is an offence which is done by a person who at the time of doing it is by reason of unsoundness of mind incapable of knowing the nature of the act or that what he is doing is wrong or contrary to law.", "category": "General Exceptions", "punishment": "Not punishable - defence of insanity"},
    {"section": "IPC 85", "title": "Act of a Person Incapable due to Involuntary Intoxication", "description": "Nothing is an offence which is done by a person who at the time of doing it is by reason of intoxication incapable of knowing the nature of the act provided that the thing which intoxicated him was administered without his knowledge or will.", "category": "General Exceptions", "punishment": "Not punishable if involuntary intoxication"},
    {"section": "IPC 96", "title": "Things Done in Private Defence", "description": "Nothing is an offence which is done in the exercise of the right of private defence of body or property.", "category": "General Exceptions", "punishment": "Not punishable - right of private defence"},
    {"section": "IPC 100", "title": "When Right of Private Defence Extends to Causing Death", "description": "The right of private defence of body extends under certain circumstances to the voluntary causing of death or harm to the assailant including when there is reasonable apprehension of death or grievous hurt.", "category": "General Exceptions", "punishment": "Not punishable within limits of private defence"},
    {"section": "IPC 511", "title": "Punishment for Attempting to Commit Offences", "description": "Whoever attempts to commit an offence punishable by this Code with imprisonment for life or imprisonment and in such attempt does any act towards the commission of the offence shall be punished.", "category": "General Provisions", "punishment": "Half the longest term of imprisonment or fine or both"},
    {"section": "IPC 505", "title": "Statements Conducing to Public Mischief", "description": "Whoever makes publishes or circulates any statement rumor or report with intent to cause mutiny or offences against the state or public tranquility or to cause fear or alarm to the public shall be punished.", "category": "Offences Against Public Order", "punishment": "Up to 3 years or fine or both"},
    {"section": "IPC 494", "title": "Marrying Again during Lifetime of Husband or Wife", "description": "Whoever having a husband or wife living marries in any case in which such marriage is void by reason of its taking place during the life of such husband or wife shall be punished.", "category": "Offences Relating to Marriage", "punishment": "Up to 7 years and fine"},
    {"section": "IPC 493", "title": "Cohabitation caused by Man Deceitfully Inducing Belief of Lawful Marriage", "description": "Every man who by deceit causes any woman who is not lawfully married to him to believe that she is lawfully married to him and to cohabit with him in that belief shall be punished.", "category": "Offences Relating to Marriage", "punishment": "Up to 10 years and fine"}
]

df = pd.DataFrame(ipc_data)
print(f"✅ Dataset loaded: {len(df)} IPC sections")
print(df[['section', 'title', 'category']].head(10))

✅ Dataset loaded: 99 IPC sections
    section                                              title  \
0   IPC 299                                  Culpable Homicide   
1   IPC 300                                             Murder   
2   IPC 302                              Punishment for Murder   
3   IPC 304  Punishment for Culpable Homicide not amounting...   
4  IPC 304A                        Causing Death by Negligence   
5  IPC 304B                                        Dowry Death   
6   IPC 305      Abetment of Suicide of Child or Insane Person   
7   IPC 306                                Abetment of Suicide   
8   IPC 307                                  Attempt to Murder   
9   IPC 308                Attempt to Commit Culpable Homicide   

                category  
0  Offences Against Body  
1  Offences Against Body  
2  Offences Against Body  
3  Offences Against Body  
4  Offences Against Body  
5  Offences Against Body  
6  Offences Against Body  
7  Offences Against Bod

In [4]:
print("⏳ Loading embedding model (this takes ~1 min on first run)...")
embedder = SentenceTransformer('all-MiniLM-L6-v2')

# Create rich text for embedding
def make_embedding_text(row):
    return f"{row['section']} {row['title']} {row['description']} {row['category']} {row['punishment']}"

df['embedding_text'] = df.apply(make_embedding_text, axis=1)
corpus = df['embedding_text'].tolist()

print("⏳ Generating embeddings...")
embeddings = embedder.encode(corpus, show_progress_bar=True, batch_size=32)
embeddings = embeddings.astype(np.float32)

# Build FAISS index
dimension = embeddings.shape[1]
index = faiss.IndexFlatL2(dimension)
index.add(embeddings)

print(f"✅ FAISS index built: {index.ntotal} vectors of dimension {dimension}")

⏳ Loading embedding model (this takes ~1 min on first run)...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

...

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

⏳ Generating embeddings...


Batches:   0%|          | 0/4 [00:00<?, ?it/s]

✅ FAISS index built: 99 vectors of dimension 384


In [7]:
def retrieve_ipc_sections(query, top_k=5):
    """Retrieve top-k relevant IPC sections using semantic search."""
    query_embedding = embedder.encode([query]).astype(np.float32)
    distances, indices = index.search(query_embedding, top_k)

    results = []
    for i, (dist, idx) in enumerate(zip(distances[0], indices[0])):
        row = df.iloc[idx]
        similarity_score = float(1 / (1 + dist))  # Convert L2 distance to similarity
        results.append({
            "rank": i + 1,
            "section": row['section'],
            "title": row['title'],
            "description": row['description'],
            "category": row['category'],
            "punishment": row['punishment'],
            "similarity_score": round(similarity_score, 4)
        })
    return results


def build_rag_prompt(query, retrieved_sections):
    """Build a structured prompt for the LLM."""
    context = ""
    for r in retrieved_sections:
        context += f"""
--- {r['section']}: {r['title']} ---
Category: {r['category']}
Description: {r['description']}
Punishment: {r['punishment']}
Relevance Score: {r['similarity_score']}
"""

    prompt = f"""You are an expert Indian legal AI assistant specializing in the Indian Penal Code (IPC).

A user has described the following incident or query:
"{query}"

Based on semantic search, the following IPC sections have been retrieved as potentially relevant:
{context}

Your task:
1. Analyze which sections genuinely apply to this case and WHY
2. Rank them by relevance to the specific query
3. Explain the legal reasoning in simple terms
4. Mention any sections that do NOT apply and why
5. Provide a final recommendation

Format your response as:
**APPLICABLE SECTIONS:**
[List each applicable section with explanation]

**LEGAL ANALYSIS:**
[Detailed explanation of how the law applies]

**RECOMMENDATION:**
[Final recommendation with most relevant section(s)]

**DISCLAIMER:** This is for educational purposes only. Consult a qualified lawyer for legal advice.
"""
    return prompt


def explain_ipc_recommendation(query, top_k=5):
    """Full RAG + LLM pipeline."""
    if not query.strip():
        return "Please enter a valid query.", [], ""

    # Step 1: Retrieve
    retrieved = retrieve_ipc_sections(query, top_k=top_k)

    # Step 2: Build prompt
    prompt = build_rag_prompt(query, retrieved)

    # Step 3: LLM Inference via Groq
    try:
        response = client.chat.completions.create(
            model="llama3-70b-8192",
            messages=[
                {
                    "role": "system",
                    "content": "You are a precise Indian legal AI assistant. Always explain clearly, cite specific IPC sections, and provide structured legal analysis."
                },
                {
                    "role": "user",
                    "content": prompt
                }
            ],
            temperature=0.2,
            max_tokens=1500
        )
        llm_response = response.choices[0].message.content
    except Exception as e:
        llm_response = f"⚠️ LLM Error: {str(e)}\n\nFalling back to retrieved sections only."

    return llm_response, retrieved, prompt


# Test it
print("🧪 Testing pipeline...")
test_query = "A person was stabbed with a knife during a fight"
response, sections, _ = explain_ipc_recommendation(test_query)
print("\n📋 TOP RETRIEVED SECTIONS:")
for s in sections[:3]:
    print(f"  {s['section']}: {s['title']} (score: {s['similarity_score']})")
print("\n🤖 LLM RESPONSE:")
print(response[:500] + "...")
print("\n✅ Pipeline working!")

🧪 Testing pipeline...

📋 TOP RETRIEVED SECTIONS:
  IPC 324: Voluntarily Causing Hurt by Dangerous Weapons (score: 0.4573)
  IPC 326: Voluntarily Causing Grievous Hurt by Dangerous Weapons (score: 0.44)
  IPC 323: Punishment for Voluntarily Causing Hurt (score: 0.4192)

🤖 LLM RESPONSE:
⚠️ LLM Error: Error code: 400 - {'error': {'message': 'The model `llama3-70b-8192` has been decommissioned and is no longer supported. Please refer to https://console.groq.com/docs/deprecations for a recommendation on which model to use instead.', 'type': 'invalid_request_error', 'code': 'model_decommissioned'}}

Falling back to retrieved sections only....

✅ Pipeline working!


In [16]:
import gradio as gr

custom_css = """
.gradio-container {
    font-family: 'Georgia', 'Times New Roman', serif;
    background: #f0f4f8;
    min-height: 100vh;
}
.main-header {
    text-align: center;
    padding: 28px 20px;
    background: linear-gradient(135deg, #ffffff, #e8f0fe);
    border-radius: 16px;
    margin-bottom: 24px;
    border: 1.5px solid #c5d5f5;
    box-shadow: 0 4px 20px rgba(66, 133, 244, 0.1);
}
.tag-pill {
    background: #e8f0fe;
    color: #1a73e8;
    padding: 4px 12px;
    border-radius: 20px;
    font-size: 0.78em;
    margin: 3px;
    display: inline-block;
    border: 1px solid #c5d5f5;
    font-family: monospace;
}
.arch-box {
    background: #ffffff;
    border: 1.5px solid #dde8f8;
    border-radius: 12px;
    padding: 16px;
    font-size: 0.85em;
    color: #334;
    box-shadow: 0 2px 8px rgba(0,0,0,0.05);
}
.stat-box {
    background: #ffffff;
    border: 1.5px solid #dde8f8;
    border-radius: 12px;
    padding: 16px;
    font-size: 0.85em;
    color: #334;
    box-shadow: 0 2px 8px rgba(0,0,0,0.05);
    margin-top: 12px;
}
footer { display: none !important; }
"""

SAMPLE_QUERIES = [
    "A man stabbed his neighbor with a knife after an argument over property",
    "An employer has not paid salary for 3 months and is threatening the employee",
    "Someone is sending threatening messages and following a woman to her workplace every day",
    "A group of 6 people broke into a house at night and stole jewelry and electronics",
    "A husband and in-laws are demanding dowry and physically abusing the wife",
    "A driver ran a red light at high speed and hit a pedestrian causing death",
    "Someone posted fake defamatory news about a businessman on social media",
    "A government official demanded bribe to process a building permit",
    "A person forged signatures on a property sale document to steal land",
    "A man married a second woman without divorcing the first wife"
]


def format_retrieved_table(retrieved_sections):
    if not retrieved_sections:
        return ""
    table_md = "| Rank | Section | Title | Category | Score |\n"
    table_md += "|------|---------|-------|----------|-------|\n"
    for r in retrieved_sections:
        score = r['similarity_score']
        score_str = f"🟢 {score}" if score > 0.3 else (f"🟡 {score}" if score > 0.2 else f"🔴 {score}")
        table_md += f"| {r['rank']} | **{r['section']}** | {r['title']} | {r['category']} | {score_str} |\n"
    return table_md


def format_retrieved_details(retrieved_sections):
    if not retrieved_sections:
        return ""
    details = ""
    for r in retrieved_sections[:3]:
        details += f"""### 📌 {r['section']}: {r['title']}
**Category:** `{r['category']}`
**Relevance Score:** `{r['similarity_score']}`
**Description:** {r['description']}
**Punishment:** ⚖️ {r['punishment']}

---
"""
    return details


def run_analysis(query, top_k, progress=gr.Progress()):
    if not query.strip():
        return (
            "⚠️ Please enter a query to analyze.",
            "No sections retrieved yet.",
            "No details available.",
            "No query entered.",
            gr.update(visible=False)
        )
    progress(0.1, desc="🔍 Encoding query...")
    progress(0.3, desc="📚 Retrieving relevant IPC sections...")
    llm_response, retrieved, prompt = explain_ipc_recommendation(query, top_k=int(top_k))
    progress(0.7, desc="🤖 LLM analyzing sections...")
    retrieved_table = format_retrieved_table(retrieved)
    retrieved_details = format_retrieved_details(retrieved)
    progress(1.0, desc="✅ Done!")
    return (
        llm_response,
        retrieved_table,
        retrieved_details,
        prompt,
        gr.update(visible=True)
    )


with gr.Blocks(css=custom_css, title="⚖️ IPC AI Advisor") as demo:

    gr.HTML("""
    <div class="main-header">
        <h1 style="color:#1a3c6e; font-size:2em; margin:0; letter-spacing:-0.5px;">⚖️ IPC Section AI Advisor</h1>
        <h2 style="color:#4a6fa5; font-size:1.1em; margin:8px 0; font-weight:normal;">
            Explainable Recommendation using RAG + LLaMA 3
        </h2>
        <p style="color:#7a90b0; margin:6px 0; font-size:0.85em;">
            Describe a legal incident and get IPC section recommendations with full AI reasoning
        </p>
        <div style="margin-top:12px;">
            <span class="tag-pill">FAISS Vector Search</span>
            <span class="tag-pill">Sentence Transformers</span>
            <span class="tag-pill">LLaMA 3-70B · Groq</span>
            <span class="tag-pill">100 IPC Sections</span>
            <span class="tag-pill">Explainable AI</span>
        </div>
    </div>
    """)

    with gr.Row():
        with gr.Column(scale=2):
            gr.Markdown("### 📝 Describe the Incident")
            query_input = gr.Textbox(
                label="",
                placeholder="Example: A person was attacked with a knife during a robbery at night...",
                lines=4
            )
            top_k_slider = gr.Slider(
                minimum=3, maximum=10, value=5, step=1,
                label="🔢 Number of IPC sections to retrieve"
            )
            with gr.Row():
                analyze_btn = gr.Button("🔍 Analyze & Recommend", variant="primary", size="lg")
                clear_btn = gr.Button("🗑️ Clear", variant="secondary")

            gr.Markdown("### 💡 Sample Queries — Click to Load")
            sample_btns = []
            for i in range(0, len(SAMPLE_QUERIES), 2):
                with gr.Row():
                    b1 = gr.Button(f"📌 {SAMPLE_QUERIES[i][:58]}...", size="sm", variant="secondary")
                    sample_btns.append((b1, SAMPLE_QUERIES[i]))
                    if i + 1 < len(SAMPLE_QUERIES):
                        b2 = gr.Button(f"📌 {SAMPLE_QUERIES[i+1][:58]}...", size="sm", variant="secondary")
                        sample_btns.append((b2, SAMPLE_QUERIES[i+1]))

        with gr.Column(scale=1):
            gr.HTML("""
            <p style="font-weight:600; color:#1a3c6e; font-size:1em; margin-bottom:8px;">🏗️ How It Works</p>
            <div class="arch-box">
                <div style="margin-bottom:10px;">
                    <span style="color:#34a853; font-size:1.1em;">●</span>
                    <b style="color:#1a3c6e;"> Step 1 — Query Encoding</b><br>
                    <span style="color:#666; font-size:0.88em; margin-left:18px;">
                        all-MiniLM-L6-v2 converts your text into a 384-dimensional semantic vector
                    </span>
                </div>
                <div style="margin-bottom:10px;">
                    <span style="color:#4285f4; font-size:1.1em;">●</span>
                    <b style="color:#1a3c6e;"> Step 2 — FAISS Retrieval</b><br>
                    <span style="color:#666; font-size:0.88em; margin-left:18px;">
                        L2 nearest-neighbor search finds the most semantically similar IPC sections
                    </span>
                </div>
                <div style="margin-bottom:10px;">
                    <span style="color:#fbbc04; font-size:1.1em;">●</span>
                    <b style="color:#1a3c6e;"> Step 3 — RAG Prompt</b><br>
                    <span style="color:#666; font-size:0.88em; margin-left:18px;">
                        Retrieved sections are injected into a structured legal reasoning prompt
                    </span>
                </div>
                <div style="margin-bottom:10px;">
                    <span style="color:#ea4335; font-size:1.1em;">●</span>
                    <b style="color:#1a3c6e;"> Step 4 — LLM Inference</b><br>
                    <span style="color:#666; font-size:0.88em; margin-left:18px;">
                        LLaMA 3-70B reasons over the context and generates legal analysis
                    </span>
                </div>
                <div>
                    <span style="color:#9c27b0; font-size:1.1em;">●</span>
                    <b style="color:#1a3c6e;"> Step 5 — Explanation</b><br>
                    <span style="color:#666; font-size:0.88em; margin-left:18px;">
                        Applicable sections ranked with justification — fully transparent
                    </span>
                </div>
            </div>
            <div class="stat-box" style="margin-top:14px;">
                <p style="font-weight:600; color:#1a3c6e; margin:0 0 8px 0;">📊 Dataset & Model Info</p>
                <div style="color:#445; line-height:1.8; font-size:0.88em;">
                    📚 <b>100</b> IPC sections indexed<br>
                    🏷️ <b>15</b> legal categories<br>
                    🔢 <b>384</b>-dim embeddings<br>
                    🧠 <b>LLaMA 3-70B</b> via Groq<br>
                    ⚡ FAISS sub-ms retrieval
                </div>
            </div>
            """)

    with gr.Column(visible=False) as results_col:
        gr.HTML("<hr style='border:none; border-top:2px solid #dde8f8; margin:24px 0;'>")
        gr.Markdown("## 📊 Analysis Results")

        with gr.Tabs():
            with gr.Tab("⚖️ AI Legal Analysis"):
                ai_output = gr.Markdown()

            with gr.Tab("📋 Retrieved Sections (Ranked)"):
                retrieved_table_output = gr.Markdown()

            with gr.Tab("🔍 Top-3 Section Details"):
                retrieved_details_output = gr.Markdown()

            with gr.Tab("🛠️ RAG Prompt (XAI / Debug)"):
                prompt_output = gr.Textbox(
                    label="Full RAG Prompt sent to LLM",
                    lines=20,
                    interactive=False
                )

    # Events
    analyze_btn.click(
        fn=run_analysis,
        inputs=[query_input, top_k_slider],
        outputs=[ai_output, retrieved_table_output, retrieved_details_output, prompt_output, results_col],
        show_progress=True
    )
    clear_btn.click(
        fn=lambda: ("", "", "", "", "", gr.update(visible=False)),
        outputs=[query_input, ai_output, retrieved_table_output, retrieved_details_output, prompt_output, results_col]
    )
    for btn, sample_text in sample_btns:
        btn.click(fn=lambda t=sample_text: t, outputs=[query_input])

    gr.HTML("""
    <div style="text-align:center; margin-top:30px; color:#8a9ab0; font-size:0.78em; padding-bottom:20px;">
        ⚖️ For educational purposes only · Not legal advice · Built with RAG + LLaMA 3 + FAISS + Gradio
    </div>
    """)

print("✅ Light theme dashboard ready!")

/tmp/ipykernel_5539/1696539696.py:118: DeprecationWarning: The 'css' parameter in the Blocks constructor will be removed in Gradio 6.0. You will need to pass 'css' to Blocks.launch() instead.
  with gr.Blocks(css=custom_css, title="⚖️ IPC AI Advisor") as demo:


✅ Light theme dashboard ready!


In [17]:
demo.launch(
    share=True,          # Creates public URL
    debug=False,
    show_error=True,
    quiet=False
)

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://739839329ad57e1cb5.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
